# HeTra Pilot-Slice Audit (ydata-profiling, full mode)

Purpose: profile the **raw** IITM-HeTra_v2 slice used by the Option A pilot
to decide the next step — stratified re-split, Option B merge (BMD-45), or
accept pilot coverage as-is.

## Data flow (CPU-only runtime is fine — no GPU needed)
1. Pull the HF snapshot into ephemeral `/tmp` (same hardened downloader as
   the training notebook).
2. Parse **raw** VOC XMLs (ALL class names, including ones Option A drops)
   into two pandas DataFrames + cheap per-image pixel stats.
3. Full `ydata-profiling` reports (minimal=False) -> two HTML files.
4. **Only the HTMLs leave Colab** (browser download). Paste the printed
   text summary back for the go/no-go discussion.

## Decision rules (applied to the summary output)
- Missing val classes (`motorcycle/truck/bicycle`) present in train ->
  stratified re-split, no BMD-45 needed.
- Option A drop share large -> Option B merge justified.
- Tiny-box dominance -> note the mAP ceiling for dense/distant traffic.

## Runtime instructions
1. `Runtime` -> CPU (no GPU needed). `Runtime` -> `Restart and run all`.
2. If the install cell's import assert fails, `Restart runtime` once more
   (ydata-profiling sometimes needs a post-install restart), then run all.


## 1. Setup

Pinned versions for reproducibility. The import assert fails fast if the
post-install restart was skipped.

In [1]:
!pip install -q "ydata-profiling==4.18.4" huggingface_hub

import ydata_profiling, huggingface_hub, pandas, cv2
print('ydata-profiling:', ydata_profiling.__version__)
print('huggingface_hub :', huggingface_hub.__version__)
print('pandas          :', pandas.__version__)
print('cv2             :', cv2.__version__)
assert ydata_profiling.__version__.startswith('4.'), 'profiling install incomplete'


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 682.5/682.5 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 4.4 MB/s eta 0:00:00
ydata-profiling: 4.18.4
huggingface_hub : 1.28.0
pandas          : 2.2.3
cv2             : 4.14.0


/tmp/ipykernel_668/809216499.py:3: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  import ydata_profiling, huggingface_hub, pandas, cv2


## 2. Download

Anonymous HF downloads are rate-limited (HTTP 429): `max_workers=2` keeps
us polite; `snapshot_download` resumes cached files so re-running is cheap.
If retries stall, `huggingface-cli login` with a free token first.

In [2]:
from huggingface_hub import snapshot_download
from pathlib import Path

HF_REPO = "kalyan1729/trafficmanagementdataset"
HF_SUBSET = "IITM-HeTra_v2"
ALLOW_PATTERNS = ["IITM-HeTra_v2/**"]

RAW_ROOT = snapshot_download(repo_id=HF_REPO, repo_type="dataset",
                             allow_patterns=ALLOW_PATTERNS, max_workers=2)

# Robust subset resolution: never hardcode the snapshot interior.
cands = [d for d in Path(RAW_ROOT).rglob('*')
         if d.is_dir() and d.name.lower() == HF_SUBSET.lower()]
print('subset candidates:', [str(d) for d in cands])
assert cands, ('subset ' + HF_SUBSET + ' not found; top level: '
               + str([p.name for p in sorted(Path(RAW_ROOT).iterdir())][:15]))
SUBSET_ROOT = str(cands[0])
print('SUBSET_ROOT:', SUBSET_ROOT)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5678 files:   0%|          | 0/5678 [00:00<?, ?it/s]

subset candidates: ['/root/.cache/huggingface/hub/datasets--kalyan1729--trafficmanagementdataset/snapshots/040637b2ba82b364a9f67d22c996102ae47693dd/IITM-HeTra_v2']
SUBSET_ROOT: /root/.cache/huggingface/hub/datasets--kalyan1729--trafficmanagementdataset/snapshots/040637b2ba82b364a9f67d22c996102ae47693dd/IITM-HeTra_v2


## 3. Build DataFrames

Parses raw VOC (every class name kept verbatim for the drop audit) and
reads each image once for brightness/sharpness. Self-contained: everything
this cell needs is defined here, so re-running it alone is always safe.

In [3]:
import os, xml.etree.ElementTree as ET
import cv2
import numpy as np, pandas as pd
from pathlib import Path

# Contract mapping (mirrors scripts/prepare_dataset.py, Option A).
CLASSES = ["car", "motorcycle", "bus", "truck", "bicycle", "auto"]
ALIASES = {
    "motorbike": "motorcycle", "moto": "motorcycle",
    "bicycle": "bicycle", "bike": "bicycle",
    "autorickshaw": "auto", "rickshaw": "auto",
    "three_wheeler": "auto", "auto_rickshaw": "auto",
}

def contract_class(name):
    n = (name or "").lower().strip().replace(" ", "_").replace("-", "_")
    m = ALIASES.get(n, n)
    return m if m in CLASSES else None

def load_splits(subset_root):
    membership = {}
    for split, files in (("train", ("trainval.txt", "train.txt")),
                          ("val", ("val.txt",)), ("test", ("test.txt",))):
        for lf in files:
            for lp in sorted(Path(subset_root).rglob(lf)):
                for raw in lp.read_text(encoding="utf-8").splitlines():
                    stem = raw.strip().lstrip("\ufeff")
                    if stem:
                        membership[stem] = split
    return membership

base = Path(SUBSET_ROOT)
seen, xmls = set(), []
for xp in sorted(base.rglob('*.xml')) + sorted(base.rglob('*.XML')):
    key = str(xp.resolve()).lower()
    if key not in seen:
        seen.add(key)
        xmls.append(xp)
assert xmls, f'no VOC XML under {SUBSET_ROOT}; top: {[p.name for p in sorted(base.iterdir())][:15]}'
print(f'{len(xmls)} XML files found')

# HeTra keeps xmls/ and images/ in SEPARATE dirs: index images by stem.
img_index = {}
for pat in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG'):
    for q in base.rglob(pat):
        img_index.setdefault(q.stem, q)
print(f'{len(img_index)} images indexed')
assert img_index, 'no images found under subset'

membership = load_splits(SUBSET_ROOT)
ann_rows, img_rows = [], []
n_noimg = 0
for xp in xmls:
    try:
        root = ET.parse(xp).getroot()
    except ET.ParseError:
        continue
    size = root.find('size')
    if size is None:
        continue
    try:
        w, h = int(size.findtext('width') or 0), int(size.findtext('height') or 0)
    except ValueError:
        continue
    if w <= 0 or h <= 0:
        continue
    ip = None
    pt = root.findtext('path')
    if pt and Path(pt).exists():
        ip = Path(pt)
    if ip is None:
        fn = root.findtext('filename') or xp.stem
        for cand in (xp.parent / fn, xp.with_suffix('.jpg'),
                     img_index.get(Path(fn).stem), img_index.get(xp.stem)):
            if cand is not None and Path(cand).exists():
                ip = Path(cand)
                break
    if ip is None:
        n_noimg += 1
        continue
    boxes = []
    for obj in root.findall('object'):
        raw_cls = obj.findtext('name') or 'UNKNOWN'
        bb = obj.find('bndbox')
        if bb is None:
            continue
        try:
            xmin, ymin = float(bb.findtext('xmin')), float(bb.findtext('ymin'))
            xmax, ymax = float(bb.findtext('xmax')), float(bb.findtext('ymax'))
        except (TypeError, ValueError):
            continue
        bw, bh = xmax - xmin, ymax - ymin
        if bw <= 0 or bh <= 0:
            continue
        cc = contract_class(raw_cls)
        boxes.append((raw_cls, cc, (xmin + bw / 2) / w, (ymin + bh / 2) / h,
                    bw / w, bh / h))
    if not boxes:
        continue
    split = membership.get(ip.stem, 'train')
    gray = cv2.imread(str(ip), cv2.IMREAD_GRAYSCALE)
    assert gray is not None, f'unreadable image {ip}'
    brightness = float(gray.mean())
    sharpness = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    n_kept = sum(1 for b in boxes if b[1] is not None)
    img_rows.append({'image': ip.stem, 'split': split, 'n_boxes': len(boxes),
                   'n_kept': n_kept, 'n_dropped': len(boxes) - n_kept,
                   'brightness': round(brightness, 2), 'sharpness': round(sharpness, 2),
                   'width': w, 'height': h})
    for raw_cls, cc, cx, cy, bw_, bh_ in boxes:
        ann_rows.append({'image': ip.stem, 'split': split, 'raw_class': raw_cls,
                       'contract_class': cc if cc else 'DROPPED',
                       'kept': cc is not None, 'cx': round(cx, 4), 'cy': round(cy, 4),
                       'w': round(bw_, 4), 'h': round(bh_, 4),
                       'area': round(bw_ * bh_, 5),
                       'aspect': round(bw_ / bh_, 3)})

ann = pd.DataFrame(ann_rows)
img = pd.DataFrame(img_rows)
assert len(ann) > 0 and len(img) > 0, 'parsed zero rows — check XML/image layout'
print(f'ann: {ann.shape} | img: {img.shape} | xmls w/o image: {n_noimg}')
print('raw classes:', sorted(ann['raw_class'].unique()))
print('splits:', sorted(ann['split'].unique()))


1912 XML files found
1418 images indexed
ann: (9228, 11) | img: (1912, 9) | xmls w/o image: 0
raw classes: ['Auto', 'Bus', 'Car', 'Person']
splits: ['test', 'train']


## 4. Full ydata-profiling reports

Full mode (`minimal=False`) on ~15–30k annotation rows takes roughly 5–15 min
on a free CPU runtime. Only these two HTML files leave Colab.

In [4]:
assert 'ann' in dir() and 'img' in dir(), 'run cell 3 (Build DataFrames) first'
import os
from ydata_profiling import ProfileReport

os.makedirs('/tmp/hetra_profile', exist_ok=True)
rep_ann = ProfileReport(ann, title='HeTra annotations (raw VOC, all classes)',
                        minimal=False, progress_bar=True)
rep_ann.to_file('/tmp/hetra_profile/hetra_annotations_full.html')
print('wrote hetra_annotations_full.html')

rep_img = ProfileReport(img, title='HeTra images (counts + pixel stats)',
                        minimal=False, progress_bar=True)
rep_img.to_file('/tmp/hetra_profile/hetra_images_full.html')
print('wrote hetra_images_full.html')

from google.colab import files
files.download('/tmp/hetra_profile/hetra_annotations_full.html')
files.download('/tmp/hetra_profile/hetra_images_full.html')


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]



  0%|          | 0/11 [00:00<?, ?it/s]

  9%|▉         | 1/11 [00:00<00:01,  6.90it/s]

100%|██████████| 11/11 [00:00<00:00, 38.87it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

...

wrote hetra_images_full.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 5. Paste-back summary

Copy this cell's output back for the Option A vs B decision — the HTMLs stay
with you for deep inspection.

In [5]:
assert 'ann' in dir() and 'img' in dir(), 'run cell 3 (Build DataFrames) first'
import pandas as pd
pd.set_option('display.width', 160)
print('=== class x split (RAW names) ===')
print(pd.crosstab(ann['raw_class'], ann['split']))
print('\n=== class x split (CONTRACT; DROPPED = lost to Option A) ===')
print(pd.crosstab(ann['contract_class'], ann['split']))
drop_share = 1 - ann['kept'].mean()
print(f'\nOption A drop share: {drop_share:.1%} of {len(ann)} boxes')
print('\n=== box geometry quantiles ===')
print(ann[['area', 'aspect', 'w', 'h']].quantile([0.05, 0.25, 0.5, 0.75, 0.95]).round(4))
print('\n=== pixel-stat quantiles ===')
print(img[['brightness', 'sharpness', 'n_boxes', 'n_dropped']].quantile([0.05, 0.25, 0.5, 0.75, 0.95]).round(2))
print('\n=== images with zero kept boxes (invisible to Option A training) ===')
zero_kept = img[img['n_kept'] == 0]
print(f'{len(zero_kept)} / {len(img)} images ({len(zero_kept)/len(img):.1%})')


=== class x split (RAW names) ===
split      test  train
raw_class             
Auto         96    502
Bus          69    324
Car         604   3028
Person      809   3796

=== class x split (CONTRACT; DROPPED = lost to Option A) ===
split           test  train
contract_class             
DROPPED          809   3796
auto              96    502
bus               69    324
car              604   3028

Option A drop share: 49.9% of 9228 boxes

=== box geometry quantiles ===
        area  aspect       w       h
0.05  0.0050   0.296  0.0437  0.0913
0.25  0.0095   0.381  0.0625  0.1432
0.50  0.0160   0.490  0.0922  0.1826
0.75  0.0287   0.651  0.1328  0.2324
0.95  0.0636   1.095  0.2031  0.3382

=== pixel-stat quantiles ===
      brightness  sharpness  n_boxes  n_dropped
0.05       88.73    2880.40      1.0        0.0
0.25       93.65    3691.96      2.0        1.0
0.50       97.40    4214.08      5.0        2.0
0.75       99.73    4562.13      7.0        3.0
0.95      103.98    5072.84     

## 6. Handoff

1. Download the two HTMLs from the previous cell.
2. Paste cell 5's output back for review.
3. Verdict lanes: missing val classes present in train → stratified re-split;
   large drop share → Option B merge; tiny-box dominance → mAP ceiling note.